# Chapter 1 companion: the laser example.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Course sources: Mario Sznaier's lecture notes; T. Charalambous's slides; Phillips, Nagle and Chakrabortty, *Digital Control System Analysis and Design*, 4th ed., Pearson (2015).

For a > 0, an analog integral controller K/s around 1/(s+a) is stable for every K > 0.
Put the same controller on a computer (sample, hold, integrate with the
rectangle rule) and the loop degrades as the sampling period T grows, and is
unstable for T > 4/a.  This script reproduces the numbers of the chapter:

  * the analog design (a = 2, K = 1): y(t) = 1 - (1 + t) e^{-t}
  * the digital loop  u_k = u_{k-1} + K T e_{k-1},  y_{k+1} = p y_k + b u_k
    with p = e^{-aT}, b = (1 - p)/a  (exact at the samples)
  * step and disturbance peaks for several T, the size of the closed-loop
    roots, the phase margin, and the quantization step of a 12-bit converter.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

a, K = 2.0, 1.0          # plant 1/(s+a), integral gain K (K = a^2/4: critical damping)

In [ ]:
# ---------------------------------------------------------------- analog design
def y_analog(t):
    """Unit-step response of K/(s^2 + a s + K) = 1/(s+1)^2."""
    return 1 - (1 + t) * np.exp(-t)

In [ ]:
t = np.linspace(0, 20, 200001)
y = y_analog(t)
outside = np.nonzero(np.abs(y - 1) > 0.02)[0]
print("Analog design, a = 2, K = 1:  Y/R = 1/(s+1)^2")
print(f"  2% settling time                  = {t[outside[-1]]:.2f} s")
print(f"  disturbance peak t e^-t at t = 1  = {np.exp(-1):.3f}")

In [ ]:
# ---------------------------------------------------------------- digital loop
def digital_loop(T, r=1.0, d=0.0, n=None):
    """Simulate the sampled loop; returns k, y_k.  r: step reference, d: step
    disturbance at the plant input (both start at k = 0, loop at rest)."""
    p = np.exp(-a * T)
    b = (1 - p) / a
    n = n or int(np.ceil(40 / T)) + 2
    y = np.zeros(n)
    u = np.zeros(n)
    e = np.zeros(n)
    for k in range(n - 1):
        e[k] = r - y[k]
        if k >= 1:
            u[k] = u[k - 1] + K * T * e[k - 1]       # rectangle rule, one sample old
        y[k + 1] = p * y[k] + b * (u[k] + d)         # ZOH + plant, exact
    return np.arange(n), y

In [ ]:
print("\nDigital loop, same K, growing T")
print("   T (s)   step peak   disturbance peak   |lambda|   decay per second")

In [ ]:
for T in (0.1, 0.5, 1.0, 1.5, 2.0, 2.5):
    p = np.exp(-a * T)
    b = (1 - p) / a
    lam2 = p + b * K * T                           # |lambda|^2 = product of the complex roots
    _, ys = digital_loop(T)
    _, yd = digital_loop(T, r=0.0, d=1.0)
    if lam2 < 1:
        print(f"   {T:4.1f}    {ys.max():7.3f}     {yd.max():10.3f}         "
              f"{np.sqrt(lam2):6.3f}      {-np.log(np.sqrt(lam2)) / T:6.3f}")
    else:
        print(f"   {T:4.1f}    {'oscillates' if np.isclose(lam2, 1) else 'grows':>10}"
              f"                         {np.sqrt(lam2):6.3f}      {-np.log(np.sqrt(lam2)) / T + 0.0:6.3f}")

In [ ]:
# hand table for T = 1 s
T = 1.0

In [ ]:
k, ys = digital_loop(T, n=8)
print("\nT = 1 s, first samples of y_k:", np.round(ys[:7], 4))
print("Stability limit of the digital loop: T < 4/a =", 4 / a, "s")

In [ ]:
# ---------------------------------------------------------------- phase margin
# delay estimate: L = 1/(s(s+2)) with a delay of about T
wc = np.sqrt(np.sqrt(5) - 2)

In [ ]:
pm0 = 90 - np.degrees(np.arctan(wc / 2))
print(f"\nAnalog loop: crossover {wc:.3f} rad/s, phase margin {pm0:.1f} deg")
print("   T (s)   PM, delay estimate   PM of the sampled loop")

In [ ]:
for T in (0.1, 0.5, 1.0, 1.5, 2.0):
    p = np.exp(-a * T)
    b = (1 - p) / a
    w = np.linspace(1e-4, np.pi / T, 400001)
    z = np.exp(1j * w * T)
    L = K * T * b / ((z - 1) * (z - p))            # D(z) = KT/(z-1), G(z) = b/(z-p)
    i = np.argmin(np.abs(np.abs(L) - 1))
    pm = (np.degrees(np.angle(L[i])) + 360) % 360 - 180   # 180 + angle, wrapped to (-180, 180]
    pm = 0.0 if abs(pm) < 0.05 else pm
    print(f"   {T:4.1f}        {pm0 - np.degrees(wc * T):6.1f}             {pm:6.1f}")

In [ ]:
# ---------------------------------------------------------------- quantization
q = 20 / 2**12

In [ ]:
print(f"\n12-bit converter on +-10 V: q = {1000 * q:.2f} mV, rounding error <= {500 * q:.2f} mV")

In [ ]:
# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

In [ ]:
tt = np.linspace(0, 12, 600)
axes[0].plot(tt, y_analog(tt), "k--", lw=1, label="analog design")

In [ ]:
for T, c in [(0.5, "C0"), (1.0, "C1"), (1.5, "C3")]:
    k, ys = digital_loop(T, n=int(12 / T) + 1)
    axes[0].plot(k * T, ys, "o-", ms=3, color=c, label=f"samples, T = {T} s")

In [ ]:
axes[0].set(xlabel="t (s)", ylabel="y", title="Step response")
axes[0].legend(fontsize=8)

In [ ]:
for T, c in [(2.0, "C0"), (2.5, "C3")]:
    k, ys = digital_loop(T, n=int(40 / T) + 1)
    axes[1].plot(k * T, ys, "o-", ms=3, color=c, label=f"T = {T} s")

In [ ]:
axes[1].set(xlabel="t (s)", title="At and past the limit T = 4/a")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch01_laser_step.png", dpi=150)
print("\nSaved figures/ch01_laser_step.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**The laser loop on a computer.** The analog design $K/s$ around $1/(s+a)$, $a = 2$ s$^{-1}$,
is stable for every $K$. Here the same integrator runs on a computer as
$u_k = u_{k-1} + KT\,e_{k-1}$, with a zero-order hold in front of the plant.
Keep $K = 1$ and push $T$ from 0.1 s toward 2 s: the overshoot grows, and past $T = 2$ s the
output oscillates, then grows. Then change $K$ and watch the limit move to $T = a/K$.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

A = 2.0          # plant 1/(s + a), a = 2 1/s
T_END = 20.0     # seconds shown


def _digital_loop(T, K, r, d, n):
    """Samples of the sampled loop and the held control, from rest."""
    p = np.exp(-A * T)
    b = (1 - p) / A
    y = np.zeros(n + 1)
    u = np.zeros(n + 1)
    e = np.zeros(n + 1)
    for k in range(n):
        e[k] = r - y[k]
        if k >= 1:
            u[k] = u[k - 1] + K * T * e[k - 1]       # rectangle rule, error one sample old
        y[k + 1] = p * y[k] + b * (u[k] + d)         # ZOH + plant, exact at the samples
    return y, u


def explore(T=1.0, K=1.0, input="step command r"):
    disturbance = input.startswith("step disturbance")
    r, d = (0.0, 1.0) if disturbance else (1.0, 0.0)
    n = int(np.ceil(T_END / T))
    y, u = _digital_loop(T, K, r, d, n)

    # between samples the plant sees a constant input: exact exponential arcs
    tau = np.linspace(0, T, 25)
    t_c = np.concatenate([k * T + tau for k in range(n)])
    y_c = np.concatenate([np.exp(-A * tau) * y[k] + (1 - np.exp(-A * tau)) * (u[k] + d) / A
                          for k in range(n)])

    # analog design: Y/R = K/(s^2 + a s + K), Y/D = s/(s^2 + a s + K)
    t_a = np.linspace(0, T_END, 801)
    num = [1.0, 0.0] if disturbance else [K]
    _, y_a = signal.step(([*num], [1.0, A, K]), T=t_a)

    # modes of the sampled loop: lambda^2 - (1 + p) lambda + (p + b K T) = 0
    p = np.exp(-A * T)
    b = (1 - p) / A
    lam = np.roots([1.0, -(1 + p), p + b * K * T])
    rho = np.max(np.abs(lam))
    stable = rho < 1 - 1e-12

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0),
                                   gridspec_kw={"width_ratios": [2.1, 1]})
    ax1.plot(t_a, y_a, "--", color=GRAY, lw=1.5, label="analog design")
    ax1.plot(t_c, y_c, color=ORANGE, lw=1.6, label="y(t), computer in the loop")
    k_show = np.arange(n + 1)
    ax1.plot(k_show * T, y, "o", color=ORANGE, ms=4 if T >= 0.3 else 2, label="samples y(kT)")
    ax1.axhline(0 if disturbance else 1, color="k", lw=0.6)
    ax1.set_xlim(0, T_END)
    lo, hi = min(-0.2, y_a.min() - 0.1), max(1.9 if not disturbance else 0.8, y_a.max() + 0.1)
    if stable:
        lo, hi = min(lo, y.min() - 0.1), max(hi, y.max() + 0.1)
    else:
        lo, hi = -3.0, 4.0
    ax1.set_ylim(lo, hi)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("beam position y")
    what = "unit step disturbance d (r = 0)" if disturbance else "unit step command r"
    ax1.set_title(f"T = {T:g} s, K = {K:g}: {what}", fontsize=11)
    ax1.legend(fontsize=10, loc="upper right")

    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(lam.real, lam.imag, "x", color=ORANGE, ms=11, mew=2.5)
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    lim = max(1.3, 1.1 * rho)
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re $\\lambda$")
    ax2.set_ylabel("Im $\\lambda$")
    ax2.set_title("modes $\\lambda$ and the unit circle", fontsize=11)
    fig.tight_layout()

    print(f"p = e^(-aT) = {p:.4f},  b = (1 - p)/a = {b:.4f}")
    print("modes lambda =", ", ".join(f"{z.real:.3f}{z.imag:+.3f}j" for z in lam),
          f"   max |lambda| = {rho:.3f}")
    if stable:
        peak = y.max()
        label = "disturbance peak" if disturbance else "step peak"
        print(f"stable: {label} of y = {peak:.3f},  decay per second = {-np.log(rho) / T:.3f}")
    else:
        print("not stable: the samples do not decay")
    print(f"sampled loop is stable exactly when T < a/K = {A / K:.2f} s "
          "(the analog loop: every K > 0)")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(explore,
             T=FloatSlider(value=1.0, min=0.05, max=3.0, step=0.05, description="T (s)",
                           continuous_update=False),
             K=FloatSlider(value=1.0, min=0.25, max=3.0, step=0.25, description="K",
                           continuous_update=False),
             input=Dropdown(options=["step command r", "step disturbance d"],
                            value="step command r", description="input"))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 1.** These cells go with the Hands-on exercises at the end of
Chapter 1. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
# ---------------------------------------------------------------- H1
def h1_staircase(T=0.05):
    """H1: a 1 Hz sine through a sample-and-hold; measure the lag of the fundamental."""
    f = 1.0
    t = np.linspace(0, 2, 4000, endpoint=False)
    held = np.sin(2 * np.pi * f * np.floor(t / T) * T)
    # fundamental of the staircase: project on sin and cos over two whole periods
    a = 2 * np.mean(held * np.sin(2 * np.pi * f * t))
    b = 2 * np.mean(held * np.cos(2 * np.pi * f * t))
    amp, phase = np.hypot(a, b), np.degrees(np.arctan2(b, a))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, np.sin(2 * np.pi * f * t), color=GRAY, lw=1.4, label="e(t)")
    ax.step(t, held, where="post", color=ORANGE, lw=1.6, label="sample-and-hold output")
    ax.plot(t, amp * np.sin(2 * np.pi * f * t + np.radians(phase)), "--", color=BLUE, lw=1.4,
            label="its fundamental")
    ax.set_xlabel("t (s)")
    ax.set_title(f"T = {T:g} s: measured lag {-phase:.2f} deg = {-phase / 360 * 1000:.1f} ms, "
                 f"amplitude x {amp:.4f}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_euler_vs_exact(T=0.25):
    """H2: the plant y' = -4y + 4u, unit step held from y0 = 0: Euler recursion vs exact samples."""
    n = int(np.ceil(3 / T)) + 1
    k = np.arange(n)
    p = np.exp(-4 * T)
    y_exact = 1 - p ** k
    y_euler = np.zeros(n)
    for i in range(n - 1):
        y_euler[i + 1] = (1 - 4 * T) * y_euler[i] + 4 * T
    t = np.linspace(0, k[-1] * T, 600)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, 1 - np.exp(-4 * t), color=GRAY, lw=1.4, label="true y(t)")
    ax.plot(k * T, y_exact, "o", color=ORANGE, ms=7, label=f"exact samples, pole {p:.3f}")
    ax.plot(k * T, y_euler, "s--", color=TEAL, ms=5, lw=1,
            label=f"Euler recursion, pole {1 - 4 * T:.3f}")
    ax.set_ylim(-2.5, 3.5)
    ax.set_xlabel("t (s)")
    ax.set_title(f"T = {T:g} s", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_delay(c=0.5, delay="B: reading one period late"):
    """H3: tank level y' = u, held input, c = KT; program A (no delay) or B (one period late)."""
    n, r = 30, 1.0
    y = np.zeros(n + 1)
    for k in range(n):
        if delay.startswith("A"):
            y[k + 1] = y[k] + c * (r - y[k])
        else:
            y[k + 1] = y[k] + c * (r - (y[k - 1] if k >= 1 else 0.0))
    roots = np.roots([1, -(1 - c)]) if delay.startswith("A") else np.roots([1, -1, c])
    rho = max(abs(roots))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(np.arange(n + 1), y, "o-", color=ORANGE, ms=4, lw=1)
    ax1.axhline(r, color="k", lw=0.6)
    ax1.set_ylim(-1, 3)
    ax1.set_xlabel("k")
    ax1.set_ylabel("$y_k$")
    ax1.set_title(f"{delay[:1]}, c = {c:.2f}: largest |root| = {rho:.3f} "
                  f"({'decays' if rho < 1 else 'does not decay'})", fontsize=11)
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(roots.real, roots.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.6, 1.6)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_title("roots $\\lambda$", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_rounded_integrator(T=0.2, bits=12):
    """H4: u_k = u_{k-1} + K T e_{k-1} stored on a grid q = 8/2^bits; plant 1/(s+2), K = 1, r = 1."""
    a, K, r = 2.0, 1.0, 1.0
    q = 8 / 2 ** bits
    p = np.exp(-a * T)
    b = (1 - p) / a
    n = int(np.ceil(40 / T))
    y = np.zeros(n + 1)
    u = np.zeros(n + 1)
    for k in range(n):
        if k >= 1:                                  # u_k = u_{k-1} + K T e_{k-1}, rounded
            u[k] = np.round((u[k - 1] + K * T * (r - y[k - 1])) / q) * q
        y[k + 1] = p * y[k] + b * u[k]
    band = q / (2 * K * T)
    t = np.arange(n + 1) * T
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, r - y, color=ORANGE, lw=1.4, label="error e = r - y")
    ax.axhspan(-band, band, color=BLUE, alpha=0.15, label=f"stall band |e| < {band:.2e}")
    ax.axhline(0.002, color=TEAL, ls="--", lw=1, label="spec |e| = 0.002")
    ax.axhline(-0.002, color=TEAL, ls="--", lw=1)
    ax.set_ylim(-0.01, 0.03)
    ax.set_xlabel("t (s)")
    ax.set_title(f"T = {T:g} s, {bits} bits: final error {r - y[-1]:.2e}, "
                 f"gain margin 2/T = {2 / T:.1f}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    interact(h1_staircase, T=FloatSlider(value=0.05, min=0.01, max=0.3, step=0.01,
                                         description="T (s)", continuous_update=False))
    interact(h2_euler_vs_exact, T=FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05,
                                              description="T (s)", continuous_update=False))
    interact(h3_delay, c=FloatSlider(value=0.5, min=0.05, max=2.2, step=0.05, description="c = KT",
                                     continuous_update=False),
             delay=Dropdown(options=["A: no delay", "B: reading one period late"],
                            value="B: reading one period late", description="program"))
    interact(h4_rounded_integrator,
             T=FloatSlider(value=0.2, min=0.05, max=1.0, step=0.01, description="T (s)",
                           continuous_update=False),
             bits=IntSlider(value=12, min=8, max=18, description="bits", continuous_update=False))
except ImportError:
    h1_staircase()
    h2_euler_vs_exact()
    h3_delay()
    h4_rounded_integrator()